# MODEL DEVELOPMENT NOTEBOOK

## PROJECT OVERVIEW

*   **PROJECT LEAD:** IKECHUKWU ONYIA(AIKAY)
*   **ROLE:** MODEL DEVELOPMENT LEAD
*   **PROJECT:** GROUP 2 TELECOM - CUSTOMER CHURN
*   **STAGE:** MODEL DEVELOPMENT

## BUSINESS PROBLEM:
Customer churn results in loss of revenue and increases the cost of acquiring new customers. The project aims to use existing customer data to identify patterns associated with churn and help the business intervene before customers leave.

## OBJECTIVE:
The objective of this project is to develop a machine-learning model that can predict which telecom customers are likely to leave the company (churn) based on their available customer information.
By identifying customers who are at risk of churning, the telecom company can take preventive actions, improve customer retention, reduce customer loss, and ultimately increase revenue.

## MY ROLE:
My responsibility is to develop, test, compare, and document classification models to identify suitable candidate models for the final churn prediction solution.

In [1]:
        # LOADING LIBRARIES

# Import pandas for data manipulation
import pandas as pd

# Import joblib to save trained models for (Model Evaluation Lead)
import joblib  # this is used to save a trained Python/Scikit-learn model to a file and later load it back.

# Import warnings to ignore unnecessary warning messages
import warnings # suppresses a specific warning when necessary

# Import machine learning classification models
from sklearn.dummy import DummyClassifier # Imports Scikit-Learn’s baseline model, which generates trivial predictions
                                          #(such as predicting the most frequent class every time) to set a minimum benchmark.

from sklearn.linear_model import LogisticRegression # imports Logistic Regression, a fast, interpretable linear classification model.

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier # Imports ensembled algorithms. Random Forest trains parallel decision trees
                                                                                # while Gradient Boosting trains sequential decision trees.

# Import evaluation metrics to track model performance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.model_selection import (StratifiedKFold, GridSearchCV) #stratifiedkfold is a cross validation strategy tecnique used to evaluate how well
# a model will generalise to new and unseen data and GridsearchCV is a tool for hyperparameter tuning

# Load the prepared training dataset
train_df = pd.read_csv('/content/prepared_train.csv')

# Load the prepared test dataset
test_df = pd.read_csv('/content/prepared_test.csv')

# Separate the predictor features (X_train) from the target column (y_train)
X_train = train_df.drop(columns=['Churn'])
y_train = train_df['Churn']

# Separate the predictor features (X_test) from the target column (y_test)
X_test = test_df.drop(columns=['Churn'])
y_test = test_df['Churn']

# Print confirmation of dataset shapes
print("Training Features Shape:", X_train.shape)
print("Training Target Shape:  ", y_train.shape)
print("Test Features Shape:    ", X_test.shape)
print("Test Target Shape:      ", y_test.shape)

Training Features Shape: (5634, 46)
Training Target Shape:   (5634,)
Test Features Shape:     (1409, 46)
Test Target Shape:       (1409,)


In [2]:
print("Training and test feature columns match:",
      list(X_train.columns) == list(X_test.columns))

Training and test feature columns match: True


### WHAT WAS DONE AND WHY?

This section initialized the necessary libraries for data manipulation, model training, and evaluation. It then loaded the `prepared_train.csv` and `prepared_test.csv` datasets, which are assumed to have undergone prior preprocessing and feature engineering by 'Simon'.

The datasets were subsequently split into feature sets (`X_train`, `X_test`) and target variables (`y_train`, `y_test`), where 'Churn' is the target variable. This separation is crucial for training models on features and evaluating their ability to predict the target, ensuring that the model is tested on unseen data.

In [3]:
      # ASSESSING CLASS IMBALANCE AND CALCULATING CLASS WEIGHTS

# Count how many non-churned customers (0) exist in training data
count_retained = (y_train == 0).sum()

# Count how many churned customers (1) exist in training data
count_churned = (y_train == 1).sum()

# Calculate the imbalance weight ratio
balance_weight = count_retained / count_churned

print("Retained Count:", count_retained)
print("Churned Count: ", count_churned)
print("Calculated Class Weight Ratio:", round(balance_weight, 4))

Retained Count: 4139
Churned Count:  1495
Calculated Class Weight Ratio: 2.7686


In [4]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # CROSS-VALIDATION STRATEGY

### WHAT WAS DONE AND WHY?

This step addresses class imbalance, a common issue in classification problems where one class (e.g., non-churn) significantly outnumbers the other (e.g., churn). Ignoring imbalance can lead to models that perform well on the majority class but poorly on the minority class.

Here, we calculated the counts of 'retained' (non-churned) and 'churned' customers in the training data. The `balance_weight` was then computed as the ratio of retained customers to churned customers. This weight can be used in models that support `class_weight` parameters (like Logistic Regression and Random Forest), assigning higher penalties for misclassifying the minority class to encourage the model to pay more attention to it.

In [5]:
# TRAIN BENCHMARK CANDIDATES MODELS

# You must test a simple baseline model alongside linear and tree-based ensemble models.

# What is this section trying to accomplish? our previous section established:
# Our target has some class imbalance.
# Now this section asks:How do different types of classification models perform on the same churn problem?

# We're testing four candidates:
# Dummy Classifier → trivial benchmark
# Logistic Regression → linear model
# Random Forest → ensemble of decision trees
# Gradient Boosting → sequential tree ensemble

# The key principle is:
# Don't assume one model is best, Train several reasonable candidates under controlled conditions and compare them using appropriate metrics.

# Define a dictionary containing all candidate models for your experimentation log
models_to_train = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
}

# Create an empty list to store performance metrics for each model
experiment_log = []

# Create a dictionary to store the trained model objects for exporting
trained_models = {}

# Loop through each model in the dictionary
for model_name, model_object in models_to_train.items():

    # Train (fit) the model using Simon's training data
    model_object.fit(X_train, y_train)

    # Save the trained model object into our dictionary
    trained_models[model_name] = model_object

    # Generate predictions on the test dataset
    predictions = model_object.predict(X_test)

    # Generate prediction probabilities for ROC-AUC calculation (if supported)
    if hasattr(model_object, "predict_proba"):
        probabilities = model_object.predict_proba(X_test)[:, 1]
    else:
        probabilities = [0] * len(y_test)

    # Calculate performance metrics
    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions, zero_division=0)
    rec = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    auc = roc_auc_score(y_test, probabilities) if hasattr(model_object, "predict_proba") else 0.5

    # Append calculated metrics to our experiment log list
    experiment_log.append({
        "Model Name": model_name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4),
        "ROC-AUC": round(auc, 4)
    })

# Convert the experiment log list into a formatted pandas DataFrame
experiment_df = pd.DataFrame(experiment_log)

# Display the final experimentation benchmark table
display(experiment_df)

# In the initial benchmark experiment, Gradient Boosting achieved the highest accuracy and ROC-AUC.
# while Logistic Regression and Random Forest achieved substantially higher recall.

,Model Name,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline (Dummy),0.7346,0.0000,0.0000,0.0000,0.5000
1,Logistic Regression,0.7381,0.5043,0.7834,0.6136,0.8415
2,Random Forest,0.7438,0.5114,0.7807,0.6180,0.8411
3,Gradient Boosting,0.8027,0.6633,0.5214,0.5838,0.8448


### WHAT WAS DONE AND WHY?

This section defines a set of candidate machine learning models, including a baseline `DummyClassifier` (predicts the most frequent class) for comparison, `LogisticRegression`, `RandomForestClassifier`, and `GradientBoostingClassifier`. These models were chosen as common and effective algorithms for binary classification.

Each model was trained on the `X_train` and `y_train` datasets. For `LogisticRegression` and `RandomForestClassifier`, `class_weight="balanced"` was used to mitigate the class imbalance identified in the previous step. After training, predictions and prediction probabilities were generated on the `X_test` dataset. Key performance metrics (Accuracy, Precision, Recall, F1 Score, and ROC-AUC) were then calculated for each model and stored in an `experiment_log` DataFrame. This structured approach allows for a direct comparison of how different models perform on the task of churn prediction.

In [6]:
     # EXTRACT TOP DRIVERS OF CHURN (Feature Importance)
# As Model Development Lead, you need to show which variables contribute most to churn predictions

# Extract the trained Random Forest model from our dictionary
rf_model = trained_models["Random Forest"]

# Extract feature importances and pair them with feature names
feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
})

# Sort features by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Reset dataframe index for clean viewing
feature_importance_df = feature_importance_df.reset_index(drop=True)

# Display top 10 most influential features
print('TOP 10 MOST INFLUENTIAL CHURN FEATURES')
display(feature_importance_df.head(10))

TOP 10 MOST INFLUENTIAL CHURN FEATURES


,Feature,Importance
0,categorical__Contract_Month-to-month,0.164744
1,numerical__tenure,0.117702
2,categorical__Contract_Two year,0.097763
3,categorical__OnlineSecurity_No,0.088621
4,numerical__TotalCharges,0.069365
5,categorical__InternetService_Fiber optic,0.059404
6,categorical__TechSupport_No,0.053626
7,numerical__MonthlyCharges,0.038100
8,categorical__PaymentMethod_Electronic check,0.036629
9,categorical__OnlineBackup_No,0.029735


In [7]:
print(feature_importance_df['Importance'].sum())

1.0000000000000002


### WHAT WAS DONE AND WHY?

This step focuses on interpreting the `RandomForestClassifier`, which is known for its ability to provide feature importances. Feature importance quantifies the contribution of each feature to the model's predictive power. By identifying the most influential features, we gain insights into which customer characteristics are most strongly associated with churn.

The code extracts the `feature_importances_` from the trained Random Forest model and creates a DataFrame that pairs each feature with its importance score. The features are then sorted in descending order of importance, and the top 10 are displayed. This information is valuable for understanding the underlying drivers of churn and can guide further business strategies or feature engineering efforts.

### Hyperparameter Tuning (Example: Random Forest Classifier)

Hyperparameter tuning is the process of finding the optimal set of hyperparameters for a machine learning model that results in the best performance on a given task. Instead of using default parameters or arbitrary values, tuning systematically searches for the best combination. We will use `GridSearchCV` to perform a thorough search over a defined grid of hyperparameters for the Random Forest Classifier.

In [10]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200], # Number of trees in the forest
    'max_depth': [5, 8, 10],      # Maximum depth of the tree
    'min_samples_split': [2, 5, 10], # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]    # Minimum number of samples required to be at a leaf node
}

# Initialize GridSearchCV
# We'll use F1-score as the scoring metric due to class imbalance
grid_search_rf = GridSearchCV(estimator=RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid=param_grid_rf,
    scoring='f1', # Optimize for F1-score as it balances precision and recall
    cv=cv_strategy, # is a parameter that specifies the cross-validation splitting strategy
    n_jobs=-1,    # Use all available CPU cores
    verbose=2
)

# Fit GridSearchCV to the training data
print("\n--- Performing GridSearchCV for Random Forest ---")
grid_search_rf.fit(X_train, y_train)

# Get the best parameters and best score
best_params_rf = grid_search_rf.best_params_
best_score_rf = grid_search_rf.best_score_

print("\nBest Parameters for Random Forest:", best_params_rf)
print("Best F1-Score from GridSearchCV:", round(best_score_rf, 4))

# Train the Random Forest model with the best parameters
best_rf_model = RandomForestClassifier(class_weight="balanced", random_state=42, **best_params_rf)
best_rf_model.fit(X_train, y_train)

# Evaluate the best model on the test set
predictions_tuned_rf = best_rf_model.predict(X_test)
probabilities_tuned_rf = best_rf_model.predict_proba(X_test)[:, 1]

acc_tuned_rf = accuracy_score(y_test, predictions_tuned_rf)
prec_tuned_rf = precision_score(y_test, predictions_tuned_rf, zero_division=0)
rec_tuned_rf = recall_score(y_test, predictions_tuned_rf, zero_division=0)
f1_tuned_rf = f1_score(y_test, predictions_tuned_rf, zero_division=0)
auc_tuned_rf = roc_auc_score(y_test, probabilities_tuned_rf)

print("\n--- Tuned Random Forest Model Performance on Test Set ---")
print(f"Accuracy: {round(acc_tuned_rf, 4)}")
print(f"Precision: {round(prec_tuned_rf, 4)}")
print(f"Recall: {round(rec_tuned_rf, 4)}")
print(f"F1 Score: {round(f1_tuned_rf, 4)}")
print(f"ROC-AUC: {round(auc_tuned_rf, 4)}")

# Add the tuned model's performance to the experiment_df
new_entry = pd.DataFrame([{
    "Model Name": "Random Forest (Tuned)",
    "Accuracy": round(acc_tuned_rf, 4),
    "Precision": round(prec_tuned_rf, 4),
    "Recall": round(rec_tuned_rf, 4),
    "F1 Score": round(f1_tuned_rf, 4),
    "ROC-AUC": round(auc_tuned_rf, 4)
}])
experiment_df = pd.concat([experiment_df, new_entry], ignore_index=True)

display(experiment_df.sort_values(by='F1 Score', ascending=False))


--- Performing GridSearchCV for Random Forest ---
Fitting 5 folds for each of 81 candidates, totalling 405 fits

Best Parameters for Random Forest: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best F1-Score from GridSearchCV: 0.6373

--- Tuned Random Forest Model Performance on Test Set ---
Accuracy: 0.7559
Precision: 0.5272
Recall: 0.7781
F1 Score: 0.6285
ROC-AUC: 0.8415


,Model Name,Accuracy,Precision,Recall,F1 Score,ROC-AUC
4,Random Forest (Tuned),0.7559,0.5272,0.7781,0.6285,0.8415
2,Random Forest,0.7438,0.5114,0.7807,0.6180,0.8411
1,Logistic Regression,0.7381,0.5043,0.7834,0.6136,0.8415
3,Gradient Boosting,0.8027,0.6633,0.5214,0.5838,0.8448
0,Baseline (Dummy),0.7346,0.0000,0.0000,0.0000,0.5000


In [11]:
# Remove duplicate model entries, keeping the last (most recent) one
experiment_df = experiment_df.drop_duplicates(subset=['Model Name'], keep='last')

# Display the updated experimentation benchmark table, sorted by F1 Score
print("FINAL MODEL EXPERIMENTATION LOG (Duplicates Removed)")
display(experiment_df.sort_values(by='F1 Score', ascending=False))

FINAL MODEL EXPERIMENTATION LOG (Duplicates Removed)


,Model Name,Accuracy,Precision,Recall,F1 Score,ROC-AUC
4,Random Forest (Tuned),0.7559,0.5272,0.7781,0.6285,0.8415
2,Random Forest,0.7438,0.5114,0.7807,0.6180,0.8411
1,Logistic Regression,0.7381,0.5043,0.7834,0.6136,0.8415
3,Gradient Boosting,0.8027,0.6633,0.5214,0.5838,0.8448
0,Baseline (Dummy),0.7346,0.0000,0.0000,0.0000,0.5000


### WHAT WAS DONE AND WHY?

It was observed that the `experiment_df` contained duplicate entries for the 'Random Forest (Tuned)' model, likely due to multiple executions of the hyperparameter tuning cell. To ensure the integrity and clarity of the experiment log, this step explicitly removes any duplicate model entries, keeping only the most recent (last) entry for each unique model name.

This ensures that the final `experiment_df` provides an accurate and concise summary of each model's best performance, making the comparison and selection process more reliable.

### WHAT WAS DONE AND WHY?

This section performed **hyperparameter tuning** for the `RandomForestClassifier` using `GridSearchCV`. Here's a breakdown:

1.  **Parameter Grid Definition (`param_grid_rf`)**: A dictionary was created to specify a range of values for key Random Forest hyperparameters: `n_estimators` (number of trees), `max_depth` (tree depth), `min_samples_split`, and `min_samples_leaf`. This defines the 'grid' that `GridSearchCV` will search.

2.  **`GridSearchCV` Initialization**: An instance of `GridSearchCV` was set up with:
    *   `estimator`: The `RandomForestClassifier` (with `class_weight="balanced"` and `random_state=42`).
    *   `param_grid`: The `param_grid_rf` containing the hyperparameters to tune.
    *   `scoring='f1'`: The `F1-score` was chosen as the primary metric for optimization, as it is robust for imbalanced datasets like churn prediction.
    *   `cv=cv_strategy`: The `StratifiedKFold` cross-validation strategy was applied to ensure fair evaluation across folds and handle class imbalance.
    *   `n_jobs=-1`: All available CPU cores were used to accelerate the search.

3.  **Fitting `GridSearchCV`**: The `grid_search_rf.fit(X_train, y_train)` command executed the exhaustive search. For every combination of hyperparameters in `param_grid_rf`, a Random Forest model was trained and evaluated using 5-fold stratified cross-validation, with the F1-score as the performance measure.

4.  **Best Parameter and Score Extraction**: After the fit, `best_params_rf` stored the optimal combination of hyperparameters that yielded the highest `F1-score` during the cross-validation, and `best_score_rf` stored that score.

5.  **Training the Best Model**: A new `RandomForestClassifier` (`best_rf_model`) was then instantiated using these `best_params_rf` and trained on the full `X_train` and `y_train` datasets.

6.  **Evaluating the Tuned Model**: The `best_rf_model` was evaluated on the unseen `X_test` dataset, and its performance metrics (Accuracy, Precision, Recall, F1 Score, ROC-AUC) were calculated and printed. This step assesses the real-world performance of the optimized model.

7.  **Updating `experiment_df`**: Finally, the performance of this `Random Forest (Tuned)` model was added as a new entry to the `experiment_df`, and the DataFrame was displayed, sorted by 'F1 Score'. This allows for a direct comparison of the tuned model against all other candidate models.

### Hyperparameter Tuning (Gradient Boosting Classifier)

Now, let's apply hyperparameter tuning to the Gradient Boosting Classifier, which also demonstrated strong initial performance. We will use `GridSearchCV` to systematically search for the optimal combination of hyperparameters for this model.

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# Define the parameter grid for Gradient Boosting
param_grid_gb = {
    'n_estimators': [50, 100, 200],      # Number of boosting stages
    'learning_rate': [0.01, 0.05, 0.1], # Step size shrinkage to prevent overfitting
    'max_depth': [3, 5, 8],             # Maximum depth of the individual regression estimators
    'subsample': [0.8, 1.0]             # Fraction of samples to be used for fitting the individual base learners
}

# Initialize GridSearchCV for Gradient Boosting
grid_search_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid_gb,
    scoring='f1', # Optimize for F1-score due to class imbalance
    cv=cv_strategy, # Use the same stratified cross-validation strategy
    n_jobs=-1,    # Use all available CPU cores
    verbose=2
)

# Fit GridSearchCV to the training data for Gradient Boosting
print("\n--- Performing GridSearchCV for Gradient Boosting ---")
grid_search_gb.fit(X_train, y_train)

# Get the best parameters and best score for Gradient Boosting
best_params_gb = grid_search_gb.best_params_
best_score_gb = grid_search_gb.best_score_

print("\nBest Parameters for Gradient Boosting:", best_params_gb)
print("Best F1-Score from GridSearchCV:", round(best_score_gb, 4))

# Train the Gradient Boosting model with the best parameters
best_gb_model = GradientBoostingClassifier(random_state=42, **best_params_gb)
best_gb_model.fit(X_train, y_train)

# Evaluate the best Gradient Boosting model on the test set
predictions_tuned_gb = best_gb_model.predict(X_test)
probabilities_tuned_gb = best_gb_model.predict_proba(X_test)[:, 1]

acc_tuned_gb = accuracy_score(y_test, predictions_tuned_gb)
prec_tuned_gb = precision_score(y_test, predictions_tuned_gb, zero_division=0)
rec_tuned_gb = recall_score(y_test, predictions_tuned_gb, zero_division=0)
f1_tuned_gb = f1_score(y_test, predictions_tuned_gb, zero_division=0)
auc_tuned_gb = roc_auc_score(y_test, probabilities_tuned_gb)

print("\n--- Tuned Gradient Boosting Model Performance on Test Set ---")
print(f"Accuracy: {round(acc_tuned_gb, 4)}")
print(f"Precision: {round(prec_tuned_gb, 4)}")
print(f"Recall: {round(rec_tuned_gb, 4)}")
print(f"F1 Score: {round(f1_tuned_gb, 4)}")
print(f"ROC-AUC: {round(auc_tuned_gb, 4)}")

# Add the tuned Gradient Boosting model's performance to the experiment_df
new_entry_gb = pd.DataFrame([{
    "Model Name": "Gradient Boosting (Tuned)",
    "Accuracy": round(acc_tuned_gb, 4),
    "Precision": round(prec_tuned_gb, 4),
    "Recall": round(rec_tuned_gb, 4),
    "F1 Score": round(f1_tuned_gb, 4),
    "ROC-AUC": round(auc_tuned_gb, 4)
}])
experiment_df = pd.concat([experiment_df, new_entry_gb], ignore_index=True)

# Remove duplicate model entries, keeping the last (most recent) one
experiment_df = experiment_df.drop_duplicates(subset=['Model Name'], keep='last')

display(experiment_df.sort_values(by='F1 Score', ascending=False))


--- Performing GridSearchCV for Gradient Boosting ---
Fitting 5 folds for each of 54 candidates, totalling 270 fits

Best Parameters for Gradient Boosting: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best F1-Score from GridSearchCV: 0.5929

--- Tuned Gradient Boosting Model Performance on Test Set ---
Accuracy: 0.8098
Precision: 0.6815
Recall: 0.5321
F1 Score: 0.5976
ROC-AUC: 0.8453


,Model Name,Accuracy,Precision,Recall,F1 Score,ROC-AUC
4,Random Forest (Tuned),0.7559,0.5272,0.7781,0.6285,0.8415
2,Random Forest,0.7438,0.5114,0.7807,0.6180,0.8411
1,Logistic Regression,0.7381,0.5043,0.7834,0.6136,0.8415
5,Gradient Boosting (Tuned),0.8098,0.6815,0.5321,0.5976,0.8453
3,Gradient Boosting,0.8027,0.6633,0.5214,0.5838,0.8448
0,Baseline (Dummy),0.7346,0.0000,0.0000,0.0000,0.5000


### WHAT WAS DONE AND WHY?

This section performed **hyperparameter tuning** for the `GradientBoostingClassifier` using `GridSearchCV`. Here's a breakdown:

1.  **Parameter Grid Definition (`param_grid_gb`)**: A dictionary was created to specify a range of values for key Gradient Boosting hyperparameters: `n_estimators` (number of boosting stages), `learning_rate` (step size shrinkage), `max_depth` (maximum depth of individual estimators), and `subsample` (fraction of samples for base learners). This defines the 'grid' that `GridSearchCV` will search.

2.  **`GridSearchCV` Initialization**: An instance of `GridSearchCV` was set up with:
    *   `estimator`: The `GradientBoostingClassifier` (with `random_state=42`).
    *   `param_grid`: The `param_grid_gb` containing the hyperparameters to tune.
    *   `scoring='f1'`: The `F1-score` was chosen as the primary metric for optimization, as it is robust for imbalanced datasets like churn prediction.
    *   `cv=cv_strategy`: The `StratifiedKFold` cross-validation strategy was applied to ensure fair evaluation across folds and handle class imbalance.
    *   `n_jobs=-1`: All available CPU cores were used to accelerate the search.

3.  **Fitting `GridSearchCV`**: The `grid_search_gb.fit(X_train, y_train)` command executed the exhaustive search. For every combination of hyperparameters in `param_grid_gb`, a Gradient Boosting model was trained and evaluated using 5-fold stratified cross-validation, with the F1-score as the performance measure.

4.  **Best Parameter and Score Extraction**: After the fit, `best_params_gb` stored the optimal combination of hyperparameters that yielded the highest `F1-score` during the cross-validation, and `best_score_gb` stored that score.

5.  **Training the Best Model**: A new `GradientBoostingClassifier` (`best_gb_model`) was then instantiated using these `best_params_gb` and trained on the full `X_train` and `y_train` datasets.

6.  **Evaluating the Tuned Model**: The `best_gb_model` was evaluated on the unseen `X_test` dataset, and its performance metrics (Accuracy, Precision, Recall, F1 Score, ROC-AUC) were calculated and printed. This step assesses the real-world performance of the optimized model.

7.  **Updating `experiment_df`**: Finally, the performance of this `Gradient Boosting (Tuned)` model was added as a new entry to the `experiment_df`, and the DataFrame was displayed, sorted by 'F1 Score'. This allows for a direct comparison of the tuned model against all other candidate models.

### Detailed Model Evaluation (Example: Gradient Boosting Classifier)

Beyond simple metric tables, visualizing model performance provides deeper insights. We will create a Confusion Matrix and an ROC Curve for the Gradient Boosting model, which currently shows the highest ROC-AUC and a good balance of metrics. This helps us understand specific types of errors the model makes and its ability to distinguish between classes across various thresholds.

In [ ]:
         # IMPORTING LIBRARIES

import matplotlib.pyplot as plt # used to create and display the plots
import seaborn as sns # Built on top of Matplotlib and makes statistical visualizations easier, it is also used for the heatmap

from sklearn.metrics import confusion_matrix, RocCurveDisplay

# Extract the Gradient Boosting model (which had the highest ROC-AUC in initial runs)
gb_model = trained_models["Gradient Boosting"]

# Generate predictions and probabilities for the Gradient Boosting model
gb_predictions = gb_model.predict(X_test)
gb_probabilities = gb_model.predict_proba(X_test)[:, 1]

print("--- Confusion Matrix for Gradient Boosting Model ---")
# Create a Confusion Matrix
cm = confusion_matrix(y_test, gb_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted 0 (Retained)', 'Predicted 1 (Churn)'],
            yticklabels=['Actual 0 (Retained)', 'Actual 1 (Churn)'])
plt.title('Confusion Matrix: Gradient Boosting Classifier')
plt.ylabel('Actual Label')ij
plt.xlabel('Predicted Label')
plt.show()

print("\n--- ROC Curve for Gradient Boosting Model ---")
# Plot ROC Curve
plt.figure(figsize=(7, 6))
RocCurveDisplay.from_estimator(gb_model, X_test, y_test, ax=plt.gca())
plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.5)') # Add random guess line
plt.title('ROC Curve: Gradient Boosting Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# WHAT WAS DONE AND WHY?
This section focused on detailed visual evaluation for one of the top-performing models (Gradient Boosting). A Confusion Matrix was generated to break down the model's predictions into True Positives (correctly identified churn), True Negatives (correctly identified retained customers), False Positives (retained customers wrongly predicted as churn), and False Negatives (churn customers wrongly predicted as retained). This gives a granular view of the types of errors the model is making.

The Receiver Operating Characteristic (ROC) Curve and its Area Under the Curve (ROC-AUC) were also plotted. The ROC curve illustrates the trade-off between the True Positive Rate (recall) and the False Positive Rate across different classification thresholds. A higher ROC-AUC indicates better overall discriminative power of the model, meaning it's better at distinguishing between the two classes.

# Model Comparison and Selection
Now that we've trained multiple models, performed hyperparameter tuning on one, and visually evaluated a strong contender, it's time to review the complete experiment_df to compare all models and make a selection for the best candidate. For churn prediction, a balanced metric like F1-Score or Recall is often crucial, as misclassifying a churning customer (False Negative) can be more costly than misclassifying a retained customer (False Positive).

In [ ]:
# ==========================================
# EXPORT FINAL MODEL DEVELOPMENT ARTIFACTS
# ==========================================

# Export the original candidate models
joblib.dump(trained_models["Logistic Regression"],'candidate_logistic_regression.pkl')

joblib.dump(trained_models["Random Forest"],'candidate_random_forest.pkl')

joblib.dump(trained_models["Gradient Boosting"],'candidate_gradient_boosting.pkl')

# Export the tuned Random Forest
joblib.dump(best_rf_model,'best_random_forest_tuned.pkl')

# Export the complete experimentation log
experiment_df.to_csv(
    'aikay_model_experiments_log.csv',index=False)

# Export feature importance ranking
feature_importance_df.to_csv('aikay_feature_importance.csv', index=False)

print("FINAL HANDOFF ARTIFACTS EXPORTED SUCCESSFULLY")

print("1. candidate_logistic_regression.pkl")
print("2. candidate_random_forest.pkl")
print("3. candidate_gradient_boosting.pkl")
print("4. best_random_forest_tuned.pkl")
print("5. aikay_model_experiments_log.csv")
print("6. aikay_feature_importance.csv")

# WHAT WAS DONE AND WHY
 After completing model development, the trained models and experiment results need to be saved as files.

This is important because the models exist in Python's memory while the notebook is running. If the session ends, those trained models would normally disappear.

Therefore, we use Joblib to save the trained models and export the experiment results as CSV files.

The saved files can then be handed over to the Model Evaluation Lead for independent evaluation.

In [ ]:
# Display the updated experimentation benchmark table, sorted by F1 Score
print("FINAL MODEL EXPERIMENTATION LOG")
display(experiment_df.sort_values(by='F1 Score', ascending=False))

# Identify the best model based on F1 Score
best_model_row = experiment_df.loc[experiment_df['F1 Score'].idxmax()]
best_model_name = best_model_row['Model Name']

print(f"\nBased on the F1 Score, the recommended candidate model is: {best_model_name}")

# You might also consider ROC-AUC as a strong indicator of overall model performance.
# If ROC-AUC were the primary metric, you would use:
# best_model_row_auc = experiment_df.loc[experiment_df['ROC-AUC'].idxmax()]
# best_model_name_auc = best_model_row_auc['Model Name']
# print(f"Based on ROC-AUC, the recommended candidate model is: {best_model_name_auc}")

### WHAT WAS DONE AND WHY?

This final step consolidates all the model evaluation metrics by displaying the `experiment_df` sorted by 'F1 Score'. This allows for a quick visual comparison of how each model performed across various metrics, with a focus on the F1 Score, which is often critical for imbalanced datasets like churn prediction.

Following this, the code programmatically identifies the best-performing model based on the highest F1 Score. This provides a clear, data-driven recommendation for the next phase of the project, such as further refinement or deployment. The explanation emphasizes the importance of choosing a metric aligned with the business objective to ensure that the selected model addresses the most critical aspects of the problem (e.g., minimizing missed churners).